# CEE6501 — Lecture 8.2

## Support Settlement

> when $\mathbf{u}_r \neq \mathbf{0}$

### Learning Objectives

By the end of this lecture, you will be able to:

- Explain what a **member release** is and why it matters in DSM
- Classify members by **release type** (no hinge / hinge at b / hinge at e / hinges at both ends)
- Derive modified local stiffness relations by enforcing **zero end moment** at a hinge
- Write the resulting **modified local stiffness matrix** and **modified fixed-end force vector**
- Recognize and handle **hinged joints** and why they can cause singular stiffness matrices

### Agenda

- **Part 1** — Concepts + notation
- **Part 2** — Derivations (MT = 1, 2, 3)
- **Part 3** — Hinged joints + singularity
- **Part 4** — Wrap-up

## Part 1 — Concepts and Notation

----

## These helper functions not displayed in slides

In [2087]:
def print_matrix_scaled(K, scale=1000, decimals=1, col_width=3):
    fmt = f"{{:{col_width}.{decimals}f}}"
    print(f"K = {scale:.0e} ×")
    for i, row in enumerate(K, start=1):
        row_scaled = row / scale
        row_str = " ".join(fmt.format(val) for val in row_scaled)
        print(f"{i} | {row_str}")

In [2088]:
def partition_system(K, f, f_fef, dof_restrained_1based):
    ndof = K.shape[0]

    # Convert restrained DOFs to 0-based
    restrained_dofs = sorted(int(d) - 1 for d in dof_restrained_1based)

    # Free DOFs
    free_dofs = [i for i in range(ndof) if i not in restrained_dofs]

    # Partition stiffness matrix
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    K_fr = K[np.ix_(free_dofs, restrained_dofs)]
    K_rf = K[np.ix_(restrained_dofs, free_dofs)]
    K_rr = K[np.ix_(restrained_dofs, restrained_dofs)]

    # Partition force vector
    f_f = f[free_dofs]
    f_r = f[restrained_dofs]

    f_fef_f = f_fef[free_dofs]
    f_fef_r = f_fef[restrained_dofs]

    return K_ff, K_fr, K_rf, K_rr, f_f, f_r, f_fef_f, f_fef_r, free_dofs, restrained_dofs


In [2089]:
def assemble_global_displacements(u_f, free_dofs, restrained_dofs, u_r=None):
    """
    Assemble the full global displacement vector u from partitioned results.
    """
    ndof_total = len(free_dofs) + len(restrained_dofs)
    u_global = np.zeros(ndof_total)

    if u_r is None:
        u_r = np.zeros(len(restrained_dofs))

    u_global[free_dofs] = u_f
    u_global[restrained_dofs] = u_r

    return u_global


In [2090]:
def assemble_global_forces(f_f, F_r, free_dofs, restrained_dofs):
    """
    Assemble the full global force vector f from applied loads and reactions.
    """
    ndof_total = len(free_dofs) + len(restrained_dofs)
    f_global = np.zeros(ndof_total)

    f_global[free_dofs] = f_f
    f_global[restrained_dofs] = F_r

    return f_global


In [2091]:
def print_dsm_results(u_global,
                      f_global_complete,
                      dof_restrained_1based,
                      dof_fictitious_1based,
                      disp_in_mm=False):

    ndof = len(u_global)
    rows = []

    # Handle optional fictitious list
    if dof_fictitious_1based is None:
        dof_fictitious_1based = np.array([], dtype=int)

    restrained_set = set(int(d) for d in np.atleast_1d(dof_restrained_1based))
    fictitious_set = set(int(d) for d in np.atleast_1d(dof_fictitious_1based))

    for i in range(ndof):
        dof_1based = i + 1
        mod = i % 3

        if mod == 0:
            dof_type = "u_x"
            disp = u_global[i] * (1000 if disp_in_mm else 1)
        elif mod == 1:
            dof_type = "u_y"
            disp = u_global[i] * (1000 if disp_in_mm else 1)
        else:
            dof_type = "theta"
            disp = u_global[i]  # rotations unchanged

        if dof_1based in fictitious_set:
            status = "Fictitious"
        elif dof_1based in restrained_set:
            status = "Fixed"
        else:
            status = "Free"

        rows.append([dof_1based, dof_type, status, disp, f_global_complete[i]])

    disp_unit = "mm" if disp_in_mm else "m"

    df = pd.DataFrame(
        rows,
        columns=[
            "DOF",
            "Type",
            "Status",
            f"Disp ({disp_unit} / rad)",
            "Load (kN / kN·m)"
        ]
    )

    print(df.to_string(index=False, float_format="%.3f"))

In [2092]:
def print_element(e, u_global, m_1based, T, k, Qf, disp_in_mm=False):
    idx = m_1based - 1
    u = u_global[idx]
    v = T @ u
    q = k @ v + Qf

    scale = 1000 if disp_in_mm else 1
    unit = "mm" if disp_in_mm else "m"

    u_print = u.copy()
    v_print = v.copy()
    u_print[[0,1,3,4]] *= scale
    v_print[[0,1,3,4]] *= scale

    print(f"\nElement {e}")
    print("-"*60)
    print(f"u (global) [{unit}&rad] : {np.round(u_print,3)}")
    print(f"v (local)  [{unit}&rad] : {np.round(v_print,3)}")
    print(f"q (local)  [kN&kN·m]    : {np.round(q,3)}")

---

In [2105]:
import numpy as np
import pandas as pd

# -------------------------
# Problem data (hard-coded)
# -------------------------
E = 200e6          # kPa
A = 0.0065         # m^2
I = 150e-6         # m^4
L = 5.0            # m

In [2106]:
# -------------------------
# Member 1 (MT=2)
# -------------------------
k1 = np.array([
    [260000.,     0.,     0., -260000.,     0.,     0.],
    [    0.,   720.,  3600.,      0.,  -720.,     0.],
    [    0.,  3600., 18000.,      0., -3600.,     0.],
    [-260000.,    0.,     0.,  260000.,    0.,     0.],
    [    0.,  -720., -3600.,     0.,   720.,     0.],
    [    0.,     0.,     0.,      0.,     0.,     0.]
], dtype=float)

# Local fixed-end vector for member 1 (given by the example result)
Qf1 = np.array([0., 60., 60., 0., 36., 0.], dtype=float)

# Transformation matrix for theta=90deg -> c=0, s=1
T1 = np.array([
    [ 0.,  1., 0., 0., 0., 0.],
    [-1.,  0., 0., 0., 0., 0.],
    [ 0.,  0., 1., 0., 0., 0.],
    [ 0.,  0., 0., 0., 1., 0.],
    [ 0.,  0., 0.,-1., 0., 0.],
    [ 0.,  0., 0., 0., 0., 1.]
], dtype=float)

# Mapping
m1 = np.array([1,2,3,4,5,6])

In [2107]:
# -------------------------
# Member 2 (MT=0)
# -------------------------

factor = (E * I) / (L**3)

k2 = factor * np.array(
    [
        [ (A*L**2)/I,  0.0,      0.0,     -(A*L**2)/I,  0.0,      0.0],
        [ 0.0,         12.0,     6.0*L,    0.0,        -12.0,     6.0*L],
        [ 0.0,         6.0*L,    4.0*L**2, 0.0,        -6.0*L,    2.0*L**2],
        [-(A*L**2)/I,  0.0,      0.0,      (A*L**2)/I,  0.0,      0.0],
        [ 0.0,        -12.0,    -6.0*L,    0.0,         12.0,    -6.0*L],
        [ 0.0,         6.0*L,    2.0*L**2, 0.0,        -6.0*L,    4.0*L**2],
    ],
    dtype=float,
)

# Local fixed-end force vector (MT=0, no releases)
Qf2 = np.array([0.0, 150.0, 187.5, 0.0, 150.0, -187.5], dtype=float)

# Horizontal member -> no transformation
T2 = np.eye(6, dtype=float)

# Mapping
m2 = np.array([4,5,6,7,8,9])

In [2108]:
# -------------------------
# Member 3 (MT=0)
# -------------------------
k3 = k2.copy()

Qf3 = np.zeros(6, dtype=float)
T3 = T1.copy()
m3 = np.array([10,11,12,7,8,9])

In [2109]:
k_list = [k1, k2, k3]
T_list = [T1, T2, T3]
Qf_list = [Qf1, Qf2, Qf3]
map_list = [m1, m2, m3]   # 1-based DOF maps

# Global system size (still using 1-based maps here)
ndof = int(np.max(np.concatenate(map_list)))

In [2110]:
## USER SPECIFIED

# Applied Load
F_global = np.zeros(ndof, dtype=float)
F_global[4-1] = 90.75

# Restrained DOFs
dof_restrained_1based = np.array([1, 2, 3, 10, 11], dtype=int)

# Fictitious restrained DOFs
dof_fictitious_1based = np.array([], dtype=int)

# Combine and sort
dof_restrained_1based = np.sort(
    np.concatenate((dof_restrained_1based, dof_fictitious_1based))
)

In [2111]:
K_global = np.zeros((ndof, ndof), dtype=float)
F_fef_global = np.zeros(ndof, dtype=float)

for i in range(3):

    k_local = k_list[i]
    T = T_list[i]
    Qf_local = Qf_list[i]
    dof_map = map_list[i]   # still 1-based

    # Transform to global
    K = T.T @ k_local @ T
    F_fef = T.T @ Qf_local

    # Scatter-add (convert to 0-based inside loop)
    for a in range(6):
        A = dof_map[a] - 1

        F_fef_global[A] += F_fef[a]

        for b in range(6):
            B = dof_map[b] - 1
            K_global[A, B] += K[a, b]

In [2112]:
(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    f_fef_f,
    f_fef_r,
    free_dofs,
    restrained_dofs,
) = partition_system(
    K_global,
    F_global,
    F_fef_global,
    dof_restrained_1based
)


In [2113]:
rhs = f_f - f_fef_f
u_f = np.linalg.solve(K_ff, rhs)

F_r = K_rf @ u_f + f_fef_r

u_global = assemble_global_displacements(u_f, free_dofs, restrained_dofs)
f_global_complete = assemble_global_forces(f_f, F_r, free_dofs, restrained_dofs)


In [2114]:
print_dsm_results(
    u_global,
    f_global_complete,
    dof_restrained_1based,
    dof_fictitious_1based,
    disp_in_mm=True
)

 DOF  Type Status  Disp (mm / rad)  Load (kN / kN·m)
   1   u_x  Fixed            0.000          -125.918
   2   u_y  Fixed            0.000            89.168
   3 theta  Fixed            0.000           389.592
   4   u_x   Free           91.553            90.750
   5   u_y   Free           -0.343             0.000
   6 theta   Free           -0.007             0.000
   7   u_x   Free           91.319             0.000
   8   u_y   Free           -0.811             0.000
   9 theta   Free           -0.001             0.000
  10   u_x  Fixed            0.000           -60.832
  11   u_y  Fixed            0.000           210.832
  12 theta   Free           -0.027             0.000


In [2115]:
# Element 1, 2, 3
print_element(1, u_global, m1, T1, k1, Qf1)
print_element(2, u_global, m2, T2, k2, Qf2)
print_element(3, u_global, m3, T3, k3, Qf3)



Element 1
------------------------------------------------------------
u (global) [m&rad] : [ 0.     0.     0.     0.092 -0.    -0.007]
v (local)  [m&rad] : [ 0.     0.     0.    -0.    -0.092 -0.007]
q (local)  [kN&kN·m]    : [ 89.168 125.918 389.592 -89.168 -29.918   0.   ]

Element 2
------------------------------------------------------------
u (global) [m&rad] : [ 0.092 -0.    -0.007  0.091 -0.001 -0.001]
v (local)  [m&rad] : [ 0.092 -0.    -0.007  0.091 -0.001 -0.001]
q (local)  [kN&kN·m]    : [  60.832   89.168    0.     -60.832  210.832 -304.158]

Element 3
------------------------------------------------------------
u (global) [m&rad] : [ 0.     0.    -0.027  0.091 -0.001 -0.001]
v (local)  [m&rad] : [ 0.     0.    -0.027 -0.001 -0.091 -0.001]
q (local)  [kN&kN·m]    : [ 210.832   60.832   -0.    -210.832  -60.832  304.158]


## PART 7 — Wrap-Up

### Wrap-Up

In this lecture, we:

- Defined **member releases** and enforced hinge conditions ($M_b=0$ and/or $M_e=0$)
- Standardized FEF notation:
  - $F_{xb}^F, F_{yb}^F, F_{mb}^F, F_{xe}^F, F_{ye}^F, F_{me}^F$
- Derived modified member relations for:
  - **MT = 1** (hinge at b)
  - **MT = 2** (hinge at e)
  - **MT = 3** (hinges at both ends)
- Discussed **hinged joints** and why they can make $\mathbf{K}$ singular

Next up:
- Secondary effects (support settlement, temperature, fabrication errors)
- How they enter DSM through equivalent nodal effects / modified $\mathbf{Q}_f$